In [ ]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom
from jax_sbgeom.flux_surfaces import ToroidalExtent
from jax_sbgeom.jax_utils import mesh_to_pyvista_mesh

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

def remove_zeros(array):
    a = onp.array(array)
    a[onp.abs(a) < 1e-14] = 0.0
    return a

from jax_sn.domain.geometric_domain import GeometricDomain, TopologicalSortedAngle
from jax_sn.solution_domain.solution_domain import FEMMatrices
from jax_sn.domain.geometric_domain import CrossSectionData
from jax_sn.source import Source
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
onp.set_printoptions(linewidth = 200)

In [ ]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain

In [ ]:
from jax_sn.solution_domain import compute_setup_for_solution_domain, transport_inverse_sweep_complete_vmap_fori



sigma  = 5
xs_0   = CrossSectionData.constant(sigma, 1, 1)
q      = 5
dim    = 1
degree = 10
verts, conn, cross_sections, sources = create_cartesian_setup(dim, [[(1.0, 5)]], cross_sections_regions=[xs_0], source_regions=[q], inhomogeneity_direction=0)
element_type                         = BasixLagrangianSimplex(degree, dim)
geometric_domain                     = GeometricDomain.from_mesh((verts, conn), element_type.face_template)
domain                               = Domain.from_domain_and_cross_sections(geometric_domain, cross_sections_indexed=cross_sections)
femmatrices, precomp_data = compute_setup_for_solution_domain(element_type, domain)
source_mapped = Source(jnp.repeat(sources, element_type.n_basis, axis=-1).reshape(-1, element_type.n_basis))
buffer_sweep = jnp.zeros((domain.n_elements, element_type.n_basis))

In [ ]:
angle_non_norm = jnp.array([1.0, 0.5,0.0])
angle_norm     = angle_non_norm / jnp.linalg.norm(angle_non_norm)
toposort = TopologicalSortedAngle.from_domain_angle(domain, angle_norm)

In [ ]:
solution = transport_inverse_sweep_complete_vmap_fori(femmatrices, domain, toposort, cross_sections.map_to_full_array(0), precomp_data, source_mapped, buffer_sweep )

In [ ]:
interp = jnp.linspace(0.0, 1.0, 1100)
cell_indices = domain.containing_cell(interp[:, None])

result = element_type.interpolate_from_real_coordinate(interp[:, None], cell_indices, domain, precomp_data, solution)


In [ ]:
from finite_elements.test_transport_operator import 

In [ ]:
import matplotlib.pyplot as plt

plt.plot(interp, result)

In [ ]:
import jax 
import jax.numpy as jnp
from jax_sn.solution_domain.transport_operator import transport_inverse_sweep_complete_vmap_fori
import jax_sn

import numpy as onp
from finite_elements.test_finite_elements import create_permutation_dimension
import pytest
def check_transport_inverse_sweep_complete_vmap(element, degree, 
                                                _vertices_conn_function, 
                                                _angle_function,  
                                                _expected_values_function, 
                                                _source_function, 
                                                _cross_section_function):
    
    
    lagrangian_element, domain, angle, toposort, femmatrices, precomputed_data, sweep_array, cross_sections, source, expected_values = _setup_all(element, degree, _vertices_conn_function, _angle_function, _expected_values_function, _source_function, _cross_section_function)


    mapped_array = cross_sections.map_to_full_array(0)

    

#    precomputed_cross_sections = cross_sections.total[]

    sweep_array = transport_inverse_sweep_complete_vmap_fori(
        femmatrices,
        domain,
        toposort,
        mapped_array, 
        precomputed_data,
        source,
        sweep_array)
    
    onp.testing.assert_allclose(sweep_array, expected_values, atol=1e-12)

    
def _setup_domain(element, degree, _vertices_conn_function):
    vertices, connectivity = _vertices_conn_function(element, degree)
    lagrangian_element     = jax_sn.solution_domain.basix_fem.BasixLagrangianSimplex(degree=degree, dimension=element)    
    domain           = jax_sn.domain.GeometricDomain.from_mesh((jnp.array(vertices)[..., :element], jnp.array(connectivity)), jnp.array(lagrangian_element.face_template))
    return lagrangian_element, domain

def _setup_all(element, degree, _vertices_conn_function, _angle_function, _expected_values_function, _source_function, _cross_section_function):
    lagrangian_element, domain = _setup_domain(element, degree, _vertices_conn_function)
    angle                      = _angle_function()    
    toposort         = jax_sn.domain.geometric_domain.TopologicalSortedAngle.from_domain_angle(domain, angle)    
    femmatrices      = jax_sn.solution_domain.FEMMatrices.from_element(lagrangian_element)    
    precomputed_data = lagrangian_element.precompute_geometric_data(domain)    
    sweep_array      = jnp.zeros((domain.n_elements, lagrangian_element.n_basis))
    cross_sections   = _cross_section_function(element, degree)
    source           = _source_function(element, degree)
    expected_values  = _expected_values_function(element, degree)
    return lagrangian_element, domain, angle, toposort, femmatrices, precomputed_data, sweep_array, cross_sections, source, expected_values

#===========================================================================================================================
#                                      StellBlanket comparison functions
#===========================================================================================================================
def _switch_element_type(dimension):
    import StellBlanket as SB
    if dimension == 1:
        return SB.Lines.Line_Element
    elif dimension == 2:
        return SB.Triangles.Triangle_Element
    elif dimension == 3:
        return SB.Tetrahedrons.Tetrahedron_Element
    else:
        raise ValueError("Dimension must be 1, 2 or 3.")
    
def _test_domain_from_sb(dimension, degree):
    import StellBlanket as SB
    material_0 = SB.Generic_Elements.Material(SB.Generic_Elements.Constant_Cross_Section(0.673))
    material_1 = SB.Generic_Elements.Material(SB.Generic_Elements.Constant_Cross_Section(2.5))
    region_sizes=  [0.8, 0.2]
    resolutions  = [10,10]
    size_other_dim = 7.54
    resolutions_other_dim = 20
    element = _switch_element_type(dimension)
    cartesian_domain = SB.Generic_Elements.Create_Slab(element, region_sizes, resolutions, [material_0, material_1], size_other_dim, resolutions_other_dim)    
    return cartesian_domain

def _arbitrary_source_values(n_elements):
    return onp.linspace(1.0, 10.0, n_elements)

def tn_order_index():
    return 4, 3

def _angle_function_indexed(tn_order, index):
    import StellBlanket as SB
    quad_set = SB.Generic_Elements.TN_Quadrature_Set(tn_order)
    return quad_set.Directions()[index:index+1,:]
    
def _matfree_nte_from_sb(dimension, degree, tn_order, angle_index):
    import StellBlanket as SB
    domain   = _test_domain_from_sb(dimension, degree)
    quad_set = SB.Generic_Elements.Quadrature_Set(onp.array(_angle_function_indexed(tn_order, angle_index)), onp.array([1.0]))

    energy_set = SB.Generic_Elements.Energy_Set([0,1])
    matfreente = SB.Generic_Elements.Create_RAM_Simulation(quad_set, energy_set, domain, degree, 0)

    source_values = _arbitrary_source_values(domain.connectivity.shape[0])
    sources = [SB.Generic_Elements.Constant_Source(_switch_element_type(dimension), val, energy_set[0]) for val in source_values]
    total_source = SB.Generic_Elements.Per_Element_Source(sources, domain)
    return matfreente, total_source
    
def _verts_conn_from_sb(dimension, degree):
    import StellBlanket as SB
    domain   = _test_domain_from_sb(dimension, degree)
    return jnp.array(domain.vertices), jnp.array(domain.connectivity)

def _source_from_sb(dimension, degree):
    nte, total_source = _matfree_nte_from_sb(dimension, degree, *tn_order_index()) # angle doesn't matter for this
    source_values = _arbitrary_source_values(nte.geometric_domain.connectivity.shape[0])
    n_values = int(nte.solution_domain.total_size / nte.geometric_domain.connectivity.shape[0])
    return jax_sn.source.Source(jnp.stack([source_values] * n_values, axis=-1))

def _cross_sections_from_sb(dimension, degree):
    nte, total_source = _matfree_nte_from_sb(dimension, degree, *tn_order_index()) # angle doesn't matter for this
    n_elements = nte.geometric_domain.connectivity.shape[0]
    cross_sections = jnp.array([nte.geometric_domain[i].Material().Cross_Section().Value_Total(nte.energy_set[0]) for i in range(nte.geometric_domain.connectivity.shape[0])])

    cross_sections = cross_sections.reshape((n_elements, 1)) # [n_elements, n_energy_groups]
    
    vals, inverse = jnp.unique(cross_sections[:,0], return_inverse=True)
    
    scattering = jnp.zeros((vals.shape[0], 1, 1,1)) # Scattering is irrelevant for transport operator test [n_elements, n_L, n_energy_groups, n_energy_groups]
    total      = jnp.array(vals).reshape((vals.shape[0], 1))
    xsdata =  jax_sn.domain.cross_sections.CrossSectionData(total, scattering)
    xsdata.assert_valid_batched()
    
    return jax_sn.domain.geometric_domain.CrossSectionIndexed(inverse, xsdata)

def _angle_function_tn():
    return _angle_function_indexed(*tn_order_index())[0]

def _n_degree_old_code(dimension):
    return [None, 3, 3, 2][dimension]


def _expected_values_from_sb(dimension, degree):
    import StellBlanket as SB
    nte, total_source = _matfree_nte_from_sb(dimension, degree, *tn_order_index())
    print(nte)
    return nte, total_source
    nte.simulation.Sweep(nte.energy_set[0], nte.quadrature_set[0], total_source)
    values = nte.global_solution.All_Solution_Values(nte.energy_set[0], nte.quadrature_set[0]).reshape(nte.geometric_domain.connectivity.shape[0], -1   )

    permutation = create_permutation_dimension(dimension, degree)

    
    
    return values[:, permutation]


nte, total_source = _expected_values_from_sb(3,2)

In [ ]:
%timeit nte.simulation.Sweep(nte.energy_set[0], nte.quadrature_set[0], total_source)